# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
# imports
import os 
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display


In [ ]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [ ]:
# set up environment
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI()    
ollama_client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # for Llama


In [ ]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
 
const users = [
  { id: 1, name: "Alice", age: 25, scores: [88, 92, 79] },
  { id: 2, name: "Bob", age: 17, scores: [55, 60, 58] },
  { id: 3, name: "Charlie", age: 30, scores: [95, 98, 100] },
  { id: 4, name: "Diana", age: 22, scores: [70, 65, 80] },
  { id: 5, name: "Eve", age: 15, scores: [40, 45, 50] },
];

function processUsers(users) {
  return users
    .filter(user => user.age >= 18)
    .map(user => ({
      ...user,
      average: user.scores.reduce((a, b) => a + b, 0) / user.scores.length,
    }))
    .sort((a, b) => b.average - a.average)
    .reduce((acc, user, index) => {
      acc[index + 1] = { name: user.name, average: user.average.toFixed(2) };
      return acc;
    }, {});
}

console.log(processUsers(users));
"""

system_prompt = "You are a helpful assistant that explains code snippets in simple terms. Please break down the code and explain its purpose and functionality step by step."

In [ ]:
# Get gpt-4o-mini to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=[{"role": "system", "content": system_prompt},
        {"role": "user", "content": question}],
    stream=True
)

result = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    if chunk.choices:
        result += chunk.choices[0].delta.content or ''
        update_display(Markdown(result), display_id=display_handle.display_id)

In [ ]:
# Get Llama 3.2 to answer

stream = ollama_client.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[{"role": "user", "content": question}],
    stream=True
)

result = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    if chunk.choices:
        result += chunk.choices[0].delta.content or ''
        update_display(Markdown(result), display_id=display_handle.display_id)